In [1]:
import torch.nn as nn
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
device = torch.device("cpu") #bc i'm doing this in aws, no gpu needed

In [2]:
model = nn.Sequential(
    nn.Flatten(), #unsurprisingly, this flattens data 0
    nn.Linear(28*28, 256), #linear layer 1
    nn.ReLU(), # 2
    nn.Linear(256, 128), #squash down 3
    nn.ReLU(), # 4
    nn.Linear(128, 10) #output 5
).to(device)
model.load_state_dict(torch.load("mlp_mnist_state_dict.pt", map_location=device))
model.eval()

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=256, bias=True)
  (2): ReLU()
  (3): Linear(in_features=256, out_features=128, bias=True)
  (4): ReLU()
  (5): Linear(in_features=128, out_features=10, bias=True)
)

In [3]:
transform = transforms.ToTensor() #this is a torchvision package for casting images to torch.Tensor

test_ds = datasets.MNIST(root="", train=False, download=True, transform=transform)
test_loader = DataLoader(test_ds, batch_size=56, shuffle=False)

In [4]:
#plan - find highest activating in l2 for features 
#i.e - if hypothesis is correct, 9 should be connected to a loop feature and a right to downward centre curve
#going to use forward hooks here
# [0]=Flatten, [1]=Linear, [2]=ReLU, [3]=Linear, [4]=ReLU, [5]=Linear
#post-relu inspection -> which neurons are most important in to causing downstream activations
#pre-relu inspection -> what's this individual neuron saying? 
#because this is a small network, i can go straight to pre-relu inspection
#if the same neurons activate with each loopy number, i have something interesting 
#plan - get mean activations for each loopy digit across all layers
#will functionalise what i already have to return activations for all layers 
#then run for each loopy number

In [5]:
#general principle at work here 
#place model into eval
#
model.eval()
n1 = model[1].out_features
n2 = model[3].out_features
n3 = model[5].out_features #just doing this so i can make a concept mask
print(n1)

256


In [6]:
acts = {}#it's safe to put this outside of the function as any changes to it will be made in the function

def save_activation(name):
    def hook(module, inp, out):
        acts[name] = out.detach()
    return hook

def get_activations_for_digit(digit, n1, n2, n3):
    

    sum_loop_1 = torch.zeros(n1, device=device)
    sum_noloop_1 = torch.zeros(n1, device=device)

    sum_loop_2 = torch.zeros(n2, device=device)
    sum_noloop_2 = torch.zeros(n2, device=device)

    sum_loop_3 = torch.zeros(n3, device=device)
    sum_noloop_3 = torch.zeros(n3, device=device)

    cnt_loop = 0
    cnt_noloop = 0 

    h1 = model[1].register_forward_hook(save_activation("z1"))
    h2 = model[3].register_forward_hook(save_activation("z2"))

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)#x is a  B*28*28 or w/e
            y = y.to(device)#y is just B

            is_loop = torch.zeros_like(y, dtype=torch.bool)#y has dimensions of y which has same dimensions as batch
            
            is_loop |= (y == digit)

            output = model(x)

            z1 = acts.pop("z1")
            z2 = acts.pop("z2")

            if is_loop.any():
                sum_loop_1 += z1[is_loop].sum(dim=0)#z1 has dimensions B, n1. a B dimension mask can be applied to it
                sum_loop_2 += z2[is_loop].sum(dim=0)#gets sum of activations for loopy numbers
                sum_loop_3 += output[is_loop].sum(dim=0)
                cnt_loop += int(is_loop.sum().item()) 

            if (~is_loop).any():
                sum_noloop_1 += z1[~is_loop].sum(dim=0)
                sum_noloop_2 += z2[~is_loop].sum(dim=0)
                sum_noloop_3 += output[~is_loop].sum(dim=0)
                cnt_noloop += int((~is_loop).sum().item())

        h1.remove()
        h2.remove()

        div_val = max(1, cnt_loop)#pointless optimisation is fun
        
        mean_loop_1 = sum_loop_1 / div_val
        mean_loop_2 = sum_loop_2 / div_val
        mean_loop_3 = sum_loop_3 / div_val

        div_val = max(1, cnt_noloop)
        mean_noloop_1 = sum_noloop_1 / div_val
        mean_noloop_2 = sum_noloop_2 / div_val
        mean_noloop_3 = sum_noloop_3 / div_val

        loop_activations = [mean_loop_1, mean_loop_2, mean_loop_3]
        noloop_activations = [mean_noloop_1, mean_noloop_2, mean_noloop_3]
        #if i remember correctly, i can calculate the scores here

        return loop_activations, noloop_activations

In [7]:
activations9, no_activations9 = get_activations_for_digit(9, n1, n2, n3)
activations8, no_activations8 = get_activations_for_digit(8, n1, n2, n3)
activations6, no_activations6 = get_activations_for_digit(6, n1, n2, n3)
activations0, no_activations0 = get_activations_for_digit(0, n1, n2, n3)

In [8]:
score1_9 = activations9[1] - no_activations9[1]
score1_8 = activations8[1] - no_activations8[1]
score1_6 = activations6[1] - no_activations6[1]
score1_0 = activations0[1] - no_activations0[1]

In [9]:
#print(score1_8)#why the hell is this all zero? this was all zeros because i messed up my indentation so the mean calculation was only performed on the last batch. there were no 8s in that so everything was just 0s. intuition to take away - if something is initially defined as 0s and stays as 0s, go back to the defintion and work through

topk = 40

top1_9 = torch.topk(score1_9, k=topk).indices.tolist()
top1_8 = torch.topk(score1_8, k=topk).indices.tolist()
top1_6 = torch.topk(score1_6, k=topk).indices.tolist()
top1_0 = torch.topk(score1_0, k=topk).indices.tolist()

In [10]:
common_neurons = set(top1_9) & set(top1_8) & set(top1_6) & set(top1_0)
print(common_neurons)#before i run this
#hypothesis - the common neurons will be the loop detector on the 2nd layer

{40, 26}


In [11]:
#further test of hypothesis - all of these common neurons will be quite high scoring in their lists
print(top1_9)
print(top1_8)
print(top1_6)
print(top1_0)

[18, 118, 85, 119, 19, 103, 60, 20, 74, 30, 109, 8, 31, 100, 29, 87, 66, 55, 44, 58, 54, 84, 62, 122, 49, 34, 41, 127, 0, 40, 90, 71, 7, 97, 28, 98, 116, 26, 89, 69]
[91, 0, 81, 76, 90, 109, 86, 122, 53, 83, 4, 50, 23, 40, 57, 43, 116, 71, 72, 77, 55, 87, 49, 47, 29, 7, 99, 10, 74, 26, 28, 41, 36, 102, 126, 66, 110, 16, 117, 42]
[62, 45, 114, 88, 60, 95, 48, 78, 84, 70, 33, 58, 27, 22, 59, 4, 46, 69, 104, 15, 24, 3, 26, 49, 56, 23, 47, 42, 36, 16, 117, 90, 125, 29, 71, 35, 64, 40, 91, 81]
[6, 16, 22, 48, 113, 97, 95, 70, 102, 35, 67, 24, 17, 106, 63, 62, 117, 69, 34, 15, 8, 61, 58, 54, 27, 14, 82, 64, 94, 40, 86, 121, 122, 44, 52, 26, 83, 73, 25, 116]


In [12]:
topactivations1_9 = score1_9[top1_9].detach().cpu().numpy() 
topactivations1_8 = score1_8[top1_8].detach().cpu().numpy() 
topactivations1_6 = score1_6[top1_6].detach().cpu().numpy() 
topactivations1_0 = score1_0[top1_0].detach().cpu().numpy() 


print(topactivations1_9)
print(topactivations1_8)
print(topactivations1_6)
print(topactivations1_0)

[4.88369    4.166001   3.6685069  3.1488547  3.0931892  2.9879766
 2.4123626  2.250197   2.0270796  2.0236096  1.9659256  1.9578844
 1.8732934  1.8294034  1.7313974  1.7060025  1.6815412  1.6247344
 1.5520656  1.5171232  1.3887541  1.3596683  1.3103102  1.2298402
 1.2183151  1.1698866  1.1581836  1.1160306  0.9372194  0.9244652
 0.86664605 0.69669175 0.6075306  0.576453   0.54954123 0.5335473
 0.5162158  0.48972702 0.48258805 0.44061035]
[2.3405573  2.3370843  2.309029   2.2927153  2.2635996  2.2056649
 1.9933603  1.933881   1.8829973  1.851346   1.8456411  1.8446511
 1.8346045  1.8069925  1.790559   1.7874905  1.7072678  1.6867166
 1.6666127  1.6480398  1.5609843  1.5380175  1.5092674  1.4925414
 1.4740849  1.4728155  1.4487773  1.38785    1.3737743  1.371445
 1.3432932  1.2398051  1.1599292  1.106105   1.070109   1.0211786
 0.79052067 0.77295256 0.71912265 0.69268584]
[5.2258525 4.8276415 4.2244835 4.1169114 4.0467935 4.0410576 3.703182
 3.6645477 3.6163583 3.313415  2.9361677 2.9271

In [13]:
#what have i proved? 
#all of the loopy numbers have common highest neurons but different bottom units

#questions to investigate
#what are the magnitudes of the common neurons? if there is a loop feature, it should be 